# Trivima — SD + LoRA Multi-View Training

Fine-tunes Stable Diffusion 1.5 with LoRA on RealEstate10K pairs.

**Run all:** Runtime → Run all (Ctrl+F9)

**Note:** This dataset (Wouter01/re10k_pixelsplat_hard) has 1 conditioning + 1 GT image per row, no real poses. The model learns single-view image-to-image generation, not true multi-view consistency. Real multi-view training needs a different dataset.

## 1. Install + GPU check

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip install -q diffusers transformers peft accelerate datasets huggingface_hub pyarrow

NVIDIA A100-SXM4-40GB, 40960 MiB


## 2. Load SD 1.5 + wrap UNet with LoRA

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler
from peft import LoraConfig, get_peft_model

device = 'cuda'

vae = AutoencoderKL.from_pretrained('stabilityai/sd-vae-ft-mse').to(device)
vae.requires_grad_(False); vae.eval()

unet = UNet2DConditionModel.from_pretrained(
    'stable-diffusion-v1-5/stable-diffusion-v1-5', subfolder='unet'
).to(device)

lora_config = LoraConfig(
    r=16, lora_alpha=16, lora_dropout=0.1,
    target_modules=['to_q', 'to_v', 'to_k', 'to_out.0'],
)
unet = get_peft_model(unet, lora_config)
unet.print_trainable_parameters()

latent_proj = nn.Linear(4, 768).to(device)

class PoseEncoder(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(16, dim), nn.SiLU(), nn.Linear(dim, dim))
    def forward(self, p):
        return self.mlp(p.reshape(p.shape[0], -1))

pose_encoder = PoseEncoder(768).to(device)

scheduler = DDPMScheduler(num_train_timesteps=1000)
scheduler.alpha_cumprod = scheduler.alphas_cumprod.to(device)
print('SD + LoRA ready')

## 3. Download RealEstate10K from HuggingFace (4 shards ≈ 1.8GB)

In [ ]:
import os
from huggingface_hub import snapshot_download

os.makedirs('/content/data/re10k_hf', exist_ok=True)
snapshot_download(
    repo_id='Wouter01/re10k_pixelsplat_hard',
    repo_type='dataset',
    local_dir='/content/data/re10k_hf',
    allow_patterns=['data/train-0000[0-3]-of-00036.parquet'],
    max_workers=4,
)
!ls -lh /content/data/re10k_hf/data/

## 4. Dataset + DataLoader

In [ ]:
import io, glob
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pyarrow.parquet as pq
import torchvision.transforms as T

class RE10KParquet(Dataset):
    def __init__(self, parquet_glob, img_size=256, max_pairs=None):
        self.files = sorted(glob.glob(parquet_glob))
        self.index = []
        for fi, p in enumerate(self.files):
            n = pq.ParquetFile(p).metadata.num_rows
            for r in range(n):
                self.index.append((fi, r))
                if max_pairs and len(self.index) >= max_pairs:
                    break
            if max_pairs and len(self.index) >= max_pairs:
                break
        self.tables = {}
        self.tf = T.Compose([
            T.Resize((img_size, img_size), T.InterpolationMode.LANCZOS),
            T.ToTensor(),
            T.Normalize([0.5]*3, [0.5]*3),
        ])
        print(f'Dataset: {len(self.index)} pairs from {len(self.files)} shards')

    def _table(self, fi):
        if fi not in self.tables:
            self.tables[fi] = pq.read_table(self.files[fi])
        return self.tables[fi]

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        fi, ri = self.index[i]
        t = self._table(fi)
        cond = t.column('conditioning_image')[ri].as_py()['bytes']
        gt   = t.column('ground_truth_image')[ri].as_py()['bytes']
        inp = self.tf(Image.open(io.BytesIO(cond)).convert('RGB'))
        tgt = self.tf(Image.open(io.BytesIO(gt)).convert('RGB'))
        pose = torch.eye(4)
        return {'input': inp, 'target': tgt, 'pose': pose}

dataset = RE10KParquet('/content/data/re10k_hf/data/train-*.parquet', img_size=256, max_pairs=10000)
loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=4, drop_last=True, pin_memory=True)
print(f'Batches/epoch: {len(loader)}')

## 5. Train (bf16, batch 32, 20 epochs, saves to /content/checkpoints_sd/)

In [ ]:
import time, numpy as np

os.makedirs('/content/checkpoints_sd', exist_ok=True)
DRIVE_DIR = '/content/drive/MyDrive/trivima_checkpoints'
HAS_DRIVE = os.path.isdir('/content/drive/MyDrive')
if HAS_DRIVE:
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print('Drive available — checkpoints will mirror to Drive')
else:
    print('Drive NOT mounted — checkpoints local only')

EPOCHS = 20
LR = 1e-4

trainable = list(unet.parameters()) + list(latent_proj.parameters()) + list(pose_encoder.parameters())
optimizer = torch.optim.AdamW([p for p in trainable if p.requires_grad], lr=LR, weight_decay=0.01)
best_loss = float('inf')

def save_ckpt(path, loss, epoch):
    torch.save({
        'unet': {k: v for k, v in unet.state_dict().items() if 'lora' in k.lower()},
        'latent_proj': latent_proj.state_dict(),
        'pose_encoder': pose_encoder.state_dict(),
        'loss': loss, 'epoch': epoch,
    }, path)

for epoch in range(EPOCHS):
    t0 = time.time()
    losses = []
    unet.train(); latent_proj.train(); pose_encoder.train()

    for bi, batch in enumerate(loader):
        inp = batch['input'].to(device, non_blocking=True)
        tgt = batch['target'].to(device, non_blocking=True)
        pose = batch['pose'].to(device, non_blocking=True)
        B = inp.shape[0]

        with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
            inp_lat = vae.encode(inp).latent_dist.sample() * 0.18215
            tgt_lat = vae.encode(tgt).latent_dist.sample() * 0.18215

        with torch.autocast('cuda', dtype=torch.bfloat16):
            pose_emb = pose_encoder(pose).unsqueeze(1)
            inp_tok  = latent_proj(inp_lat.flatten(2).transpose(1,2))
            enc_h    = torch.cat([inp_tok, pose_emb], dim=1)

            t = torch.randint(0, 1000, (B,), device=device).long()
            noise = torch.randn_like(tgt_lat)
            noisy = scheduler.add_noise(tgt_lat, noise, t)

            pred = unet(noisy, t, encoder_hidden_states=enc_h).sample
            loss = F.mse_loss(pred.float(), noise.float())

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable, 1.0)
        optimizer.step()

        losses.append(loss.item())
        if (bi+1) % 20 == 0:
            print(f'  [{epoch+1}/{EPOCHS}] {bi+1}/{len(loader)} loss={np.mean(losses[-20:]):.4f}')

    avg = float(np.mean(losses))
    dt = time.time() - t0
    print(f'Epoch {epoch+1}/{EPOCHS}: loss={avg:.4f} ({dt:.0f}s)')

    if avg < best_loss:
        best_loss = avg
        save_ckpt('/content/checkpoints_sd/best_10k.pt', best_loss, epoch)
        if HAS_DRIVE:
            save_ckpt(f'{DRIVE_DIR}/best_10k.pt', best_loss, epoch)
        print(f'  -> saved best: {best_loss:.4f}')

print(f'\nDone! Best loss: {best_loss:.4f}')

## 6. Generate + visualize

In [ ]:
import matplotlib.pyplot as plt

ckpt = torch.load('/content/checkpoints_sd/best_10k.pt', map_location=device)
missing, unexpected = unet.load_state_dict(ckpt['unet'], strict=False)
latent_proj.load_state_dict(ckpt['latent_proj'])
pose_encoder.load_state_dict(ckpt['pose_encoder'])
unet.eval(); latent_proj.eval(); pose_encoder.eval()
print(f"Loaded checkpoint, best loss: {ckpt.get('loss')}")

batch = next(iter(loader))
inp = batch['input'][:1].to(device)
gt  = batch['target'][:1].to(device)
pose = batch['pose'][:1].to(device)

with torch.no_grad():
    inp_lat = vae.encode(inp).latent_dist.sample() * 0.18215
    pose_emb = pose_encoder(pose).unsqueeze(1)
    inp_tok = latent_proj(inp_lat.flatten(2).transpose(1, 2))
    enc_h = torch.cat([inp_tok, pose_emb], dim=1)

    latent = torch.randn(1, 4, inp_lat.shape[2], inp_lat.shape[3], device=device)
    n_steps = 50
    step_size = 1000 // n_steps
    for i in range(999, -1, -step_size):
        t = torch.full((1,), i, device=device, dtype=torch.long)
        noise_pred = unet(latent, t, encoder_hidden_states=enc_h).sample
        a_t = scheduler.alpha_cumprod[i]
        a_prev = scheduler.alpha_cumprod[max(0, i - step_size)]
        pred_x0 = (latent - torch.sqrt(1 - a_t) * noise_pred) / torch.sqrt(a_t)
        pred_x0 = pred_x0.clamp(-4, 4)
        if i > 0:
            latent = torch.sqrt(a_prev) * pred_x0 + torch.sqrt(1 - a_prev) * torch.randn_like(latent)
        else:
            latent = pred_x0

    gen = vae.decode(latent / 0.18215).sample
    gen = (gen.clamp(-1, 1) + 1) / 2

def to_np(x):
    return x[0].cpu().permute(1, 2, 0).float().numpy().clip(0, 1)

inp_np = to_np((inp + 1) / 2)
gen_np = to_np(gen)
gt_np  = to_np((gt + 1) / 2)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(inp_np); axes[0].set_title('Input Photo')
axes[1].imshow(gen_np); axes[1].set_title('Generated (SD+LoRA)')
axes[2].imshow(gt_np);  axes[2].set_title('Ground Truth')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.savefig('/content/sd_lora_result.png', dpi=150)
plt.show()

mse = ((gen - (gt + 1) / 2) ** 2).mean().item()
psnr = -10 * np.log10(mse) if mse > 0 else float('inf')
print(f'PSNR: {psnr:.2f} dB')